<a href="https://colab.research.google.com/github/ratusilma/DS--dataset_prepare/blob/main/AB_Testing_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

# ==========================================
# 1. PERSIAPAN DATA (USER INTERACTION LOGS)
# ==========================================
# Catatan: Menggunakan simulasi data sampel distribusi binomial untuk
# menguji keandalan algoritma A/B Testing Kawan Kampus.

# [OPSI 1] Jika nanti menggunakan data asli hasil pengujian (Guerrilla Testing):
# df_ab = pd.read_csv('hasil_testing_kawan_kampus.csv')

# [OPSI 2] Simulasi log data pengguna untuk pengujian baseline:
np.random.seed(42)
n_samples = 100 # Total sampel observasi

# Membagi kelompok Control (Versi A) dan Treatment (Versi B)
data = {
    'user_id': [f'USR-{i+1:03d}' for i in range(n_samples)],
    'group': np.random.choice(['Control (Versi A - Urut Skor)', 'Treatment (Versi B - Urut Jarak)'], size=n_samples, p=[0.5, 0.5]),
    'converted': np.zeros(n_samples, dtype=int) # 1 = Berhasil pilih tempat, 0 = Batal
}
df_ab = pd.DataFrame(data)

# Mendistribusikan probabilitas keberhasilan berdasarkan hipotesis eksperimen
df_ab.loc[df_ab['group'] == 'Control (Versi A - Urut Skor)', 'converted'] = np.random.choice([0, 1], size=len(df_ab[df_ab['group'] == 'Control (Versi A - Urut Skor)']), p=[0.6, 0.4])
df_ab.loc[df_ab['group'] == 'Treatment (Versi B - Urut Jarak)', 'converted'] = np.random.choice([0, 1], size=len(df_ab[df_ab['group'] == 'Treatment (Versi B - Urut Jarak)']), p=[0.4, 0.6])


# ==========================================
# 2. EXPLORATORY DATA ANALYSIS (EDA) METRIK
# ==========================================
print("=== RINGKASAN PERFORMA A/B TESTING ===")
summary = df_ab.groupby('group')['converted'].agg(
    Total_User='count',
    Total_Konversi='sum'
).reset_index()

# Menghitung Conversion Rate
summary['Conversion_Rate'] = (summary['Total_Konversi'] / summary['Total_User']) * 100
summary['Conversion_Rate'] = summary['Conversion_Rate'].round(2).astype(str) + '%'

print(summary.to_string(index=False))
print("\n")


# ==========================================
# 3. UJI STATISTIK PENGAMBILAN KEPUTUSAN (Z-TEST)
# ==========================================
print("=== HASIL UJI STATISTIK ===")
successes = summary['Total_Konversi'].values
nobs = summary['Total_User'].values

# Menjalankan Proportions Z-Test
# H0: Tidak ada perbedaan Conversion Rate antara Versi A dan Versi B
# H1: Terdapat perbedaan Conversion Rate yang signifikan
stat, pval = proportions_ztest(successes, nobs)

print(f"Z-Statistic : {stat:.4f}")
print(f"P-Value     : {pval:.4f}\n")

# Menentukan batas signifikansi (Alpha = 5%)
alpha = 0.05
if pval < alpha:
    print("KESIMPULAN: Tolak H0 (P-Value < 0.05).")
    print("Terdapat perbedaan performa yang signifikan secara statistik.")
    print("Versi B (Pengurutan berdasarkan Jarak) terbukti memberikan tingkat konversi yang lebih baik untuk pengguna.")
else:
    print("KESIMPULAN: Gagal Tolak H0 (P-Value >= 0.05).")
    print("Tidak ada perbedaan performa yang signifikan antara kedua versi.")

=== RINGKASAN PERFORMA A/B TESTING ===
                           group  Total_User  Total_Konversi Conversion_Rate
   Control (Versi A - Urut Skor)          53              19          35.85%
Treatment (Versi B - Urut Jarak)          47              28          59.57%


=== HASIL UJI STATISTIK ===
Z-Statistic : -2.3725
P-Value     : 0.0177

KESIMPULAN: Tolak H0 (P-Value < 0.05).
Terdapat perbedaan performa yang signifikan secara statistik.
Versi B (Pengurutan berdasarkan Jarak) terbukti memberikan tingkat konversi yang lebih baik untuk pengguna.
